In [2]:
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [3]:
class State(TypedDict):
    messages: Annotated[list[str], add_messages]

In [6]:
def chatbot(state: State) -> str:
   last_message = state["messages"][-1] if state["messages"] else "No messages yet."
   return {
       "messages": [
           {
               "role": "assistant",
               "content": f"You said: {last_message.content}"
           }
       ]
   }

In [7]:
# create Graph
builder = StateGraph(State)
builder.add_node("chatbot", chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)


In [8]:
thread_id = "user_002"
config = {
    "configurable": {
        "thread_id": thread_id
    }
}

In [11]:
graph.invoke(
    {
        "messages": [
            ("user", "Hello, how are you?")
        ]
    },
    config=config
)

{'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='3c3d9143-5b87-406b-8dd7-32d6a8e55670'),
  AIMessage(content='You said: Hello, how are you?', additional_kwargs={}, response_metadata={}, id='a5a1ed6f-1614-440f-9225-c4379db2e5c6', tool_calls=[], invalid_tool_calls=[])]}